# Topic Hierarchy — iGEM Teams (Low → Mid → High)

Builds a three-level hierarchy for the existing **iGEM Teams** topic model
by cutting BERTopic's agglomerative merge tree at two levels:

- **mid** — auto-selected by silhouette over `[HIGH_K_MAX + 1, n_low // 3]`
- **high** — auto-selected by silhouette over `[HIGH_K_MIN, HIGH_K_MAX]`

**Inputs:** `teams_topic_model`, `teams_doc_topics.txt`, `teams_topic_names.txt`, `teams_corpus.txt`, `igem.txt`

**Outputs (in `assets/reports/`):**
- `teams_topic_hierarchy_map.tsv` — document-level mapping (`UT, low, mid, high`)
- `teams_topic_name_hierarchy.tsv` — low-level names mapped to `low, mid, high`
- `teams_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [ ]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd

from aux.paths import REPORTS_DIR, HIGH_K_MIN, HIGH_K_MAX, set_seed
from aux.hierarchy import load_hierarchy_inputs, select_hierarchy_levels, write_hierarchy_reports

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"
ID_COL = "UT"
YEAR_COL = "Year_y"
RAW_FILE = "igem.txt"
RENAME_ID_FROM = None   # source id column to rename to ID_COL (None if already named)

model, doc_topics, topic_names, corpus, raw = load_hierarchy_inputs(
    PREFIX, id_col=ID_COL, year_col=YEAR_COL, raw_filename=RAW_FILE, rename_id_from=RENAME_ID_FROM,
)
print(f"{PREFIX}: {len(doc_topics):,} docs, {len(topic_names):,} topics, "
      f"outliers={(doc_topics['low'] == -1).sum():,}")

## 1. Build hierarchy and auto-select mid / high levels

In [ ]:
corpus_texts = corpus["text"].astype(str).tolist()
hierarchy_map, sel = select_hierarchy_levels(model, corpus_texts, HIGH_K_MIN, HIGH_K_MAX)

print(f"\nHigh K = {sel['high_k']} (silhouette {sel['high_score']:.4f})  |  "
      f"Mid K = {sel['mid_k']} (silhouette {sel['mid_score']:.4f})")
print("\n--- High-level candidates ---")
display(pd.DataFrame(sel["high_scores"], columns=["high_k", "silhouette"]).sort_values("high_k"))
print("--- Mid-level candidates ---")
pd.DataFrame(sel["mid_scores"], columns=["mid_k", "silhouette"]).sort_values("mid_k")

## 2. Build and save the report tables

In [ ]:
doc_map, name_map, summary = write_hierarchy_reports(
    doc_topics, topic_names, raw, hierarchy_map,
    id_col=ID_COL, year_col=YEAR_COL, prefix=PREFIX,
)
print(f"Saved 3 hierarchy reports for {PREFIX} → {REPORTS_DIR}")
name_map.head()

## 3. Validation

In [ ]:
assert list(doc_map.columns) == [ID_COL, "low", "mid", "high"]
assert list(name_map.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(summary.columns)
assert len(doc_map) == len(doc_topics)
assert (doc_map.loc[doc_map["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= sel["high_k"] <= HIGH_K_MAX
assert sel["mid_min"] <= sel["mid_k"] <= sel["mid_max"]
print("All validation checks passed ✓")

## 4. Hierarchy quality summary

In [ ]:
outlier_pct = (doc_topics["low"] == -1).mean() * 100
print(f"Selected high-level K : {sel['high_k']}  (silhouette = {sel['high_score']:.4f})")
print(f"Selected mid-level K  : {sel['mid_k']}  (silhouette = {sel['mid_score']:.4f})")
print(f"Outlier documents     : {outlier_pct:.2f}%")
print(f"Mid-level search range: [{sel['mid_min']}, {sel['mid_max']}]")